# LABORATORIO CALIFICADO N° 07
## Fundamentos de Gestión de Datos — Semana 15
**Docente:** Pilar Rocío Sayán Mejía | **Semestre:** 2026-I

---
## CASO: Rescatando la Confianza en BancoSur
**Protagonista:** Marcos Díaz, Chief Data Officer (CDO)
**Empresa:** BancoSur — Banco regional con 45 agencias en el sur del Perú

**Tareas urgentes (30 días para la SBS):**
1. Reporte de calidad por las 5 dimensiones DAMA
2. Corregir problemas identificados con pipeline automatizado
3. Presentar marco de gobierno DAMA-DMBOK con roles y políticas

---

## PASO 1: Generación del Dataset BancoSur con Problemas de Calidad

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import re
from datetime import datetime, timedelta

np.random.seed(42)
n=400

# Generar RUC: algunos invalidos (no 11 digitos)
rucs=[]
for i in range(n):
    if np.random.random()<0.12:  # 12% invalidos
        rucs.append(str(np.random.randint(1000000,99999999)))  # 7-8 digitos
    else:
        rucs.append(str(np.random.randint(10000000000,19999999999)))

# Nombres con variantes (duplicados por ortografia)
nombres_base=['Carlos Mendoza','Maria Garcia','Luis Torres','Ana Quispe','Jorge Saenz']
nombres_variantes={
    'Carlos Mendoza':['CARLOS MENDOZA','carlos mendoza','Carlos Mendosa','Carlos Mendoza'],
    'Maria Garcia':['MARIA GARCIA','Maria García','maria garcia','Maria Garcia'],
    'Luis Torres':['LUIS TORRES','Luis Torrez','luis torres','Luis Torres'],
    'Ana Quispe':['ANA QUISPE','Ana Quipse','Ana Quispe','ANA quispe'],
    'Jorge Saenz':['JORGE SAENZ','Jorge Saenz','jorge saenz','Jorge Sáenz'],
}
all_names=[]
for _ in range(n):
    base=np.random.choice(nombres_base)
    all_names.append(np.random.choice(nombres_variantes[base]))

# Fechas: algunas muy antiguas (>5 años = desactualizadas)
fechas=[]
for i in range(n):
    if np.random.random()<0.15:  # 15% desactualizados
        d=datetime(2018,1,1)+timedelta(days=np.random.randint(0,365))
    else:
        d=datetime(2024,1,1)+timedelta(days=np.random.randint(0,500))
    fechas.append(d.strftime('%Y-%m-%d'))

df=pd.DataFrame({
    'id_cliente':[f'BS{i:05d}' for i in range(1,n+1)],
    'nombre':all_names,
    'ruc':rucs,
    'ciudad':np.random.choice(['Arequipa','Puno','Cusco','Moquegua','Tacna'],n),
    'tipo_cuenta':np.random.choice(['Ahorro','Corriente','Plazo_fijo'],n,p=[0.5,0.3,0.2]),
    'saldo':np.random.uniform(100,50000,n).round(2),
    'telefono':[str(np.random.randint(900000000,999999999)) if np.random.random()>0.18 else None for _ in range(n)],
    'email':[f'cliente{i}@mail.com' if np.random.random()>0.22 else None for i in range(n)],
    'fecha_actualizacion':fechas,
})
print(f'Dataset BancoSur: {df.shape}')
display(df.head())

## PASO 2: Reporte de Calidad — 5 Dimensiones DAMA

In [ ]:
def calcular_calidad(df):
    total=len(df)
    reporte={}

    # 1. COMPLETITUD
    nulos=df.isnull().sum().sum()
    campos_totales=total*len(df.columns)
    reporte['completitud']=round((1-nulos/campos_totales)*100,2)

    # 2. EXACTITUD (RUC = 11 digitos y empieza en 10 o 20)
    ruc_valido=df['ruc'].apply(lambda x: bool(re.match(r'^[12][0-9]{10}$',str(x))))
    reporte['exactitud_ruc']=round(ruc_valido.mean()*100,2)

    # 3. CONSISTENCIA (nombres normalizados)
    nombres_norm=df['nombre'].str.upper().str.strip()
    nombres_unicos_orig=df['nombre'].nunique()
    nombres_unicos_norm=nombres_norm.nunique()
    reporte['consistencia_nombres']=round((1-(nombres_unicos_orig-nombres_unicos_norm)/nombres_unicos_orig)*100,2)

    # 4. UNICIDAD (duplicados exactos)
    dup=df.duplicated(subset=['ruc','nombre']).sum()
    reporte['unicidad']=round((1-dup/total)*100,2)

    # 5. OPORTUNIDAD (fecha < 3 años)
    from datetime import datetime
    hoy=datetime(2026,5,1)
    fechas_ok=df['fecha_actualizacion'].apply(lambda x: (hoy-datetime.strptime(x,'%Y-%m-%d')).days < 3*365)
    reporte['oportunidad']=round(fechas_ok.mean()*100,2)

    return reporte

reporte_antes=calcular_calidad(df)
print('REPORTE DE CALIDAD BANCOSUR -- ANTES DE CORRECCIONES:')
print(f'  Completitud:             {reporte_antes["completitud"]}%')
print(f'  Exactitud (RUC):         {reporte_antes["exactitud_ruc"]}%')
print(f'  Consistencia (nombres):  {reporte_antes["consistencia_nombres"]}%')
print(f'  Unicidad (sin dup):      {reporte_antes["unicidad"]}%')
print(f'  Oportunidad (vigencia):  {reporte_antes["oportunidad"]}%')

## PASO 3: Pipeline de Corrección Automática

In [ ]:
df_corr=df.copy()

# 1. Completitud: imputar telefono y email
df_corr['telefono']=df_corr['telefono'].fillna('Sin_registro')
df_corr['email']=df_corr['email'].fillna('Sin_registro')

# 2. Exactitud: marcar RUC invalidos
df_corr['ruc_valido']=df_corr['ruc'].apply(lambda x: bool(re.match(r'^[12][0-9]{10}$',str(x))))
df_corr.loc[~df_corr['ruc_valido'],'ruc']='PENDIENTE_VERIFICACION'
print(f'RUC invalidos marcados: {(~df_corr["ruc_valido"]).sum()}')

# 3. Consistencia: normalizar nombres
df_corr['nombre']=df_corr['nombre'].str.upper().str.strip()
print(f'Nombres normalizados a mayusculas y sin espacios')

# 4. Unicidad: eliminar duplicados
antes=len(df_corr)
df_corr=df_corr.drop_duplicates(subset=['ruc','nombre'],keep='first')
despues=len(df_corr)
print(f'Duplicados eliminados: {antes-despues}')

# 5. Oportunidad: actualizar fechas antiguas
from datetime import datetime
hoy='2026-05-01'
df_corr['fecha_actualizacion']=df_corr['fecha_actualizacion'].apply(
    lambda x: hoy if (datetime(2026,5,1)-datetime.strptime(x,'%Y-%m-%d')).days > 3*365 else x)
print(f'Registros con fecha actualizada: OK')
print(f'\nDataset corregido: {df_corr.shape}')

## PASO 4: Comparación Antes vs Después

In [ ]:
reporte_despues=calcular_calidad(df_corr)

dimensiones=['Completitud','Exactitud\n(RUC)','Consistencia\n(Nombres)','Unicidad','Oportunidad']
valores_antes=[reporte_antes['completitud'],reporte_antes['exactitud_ruc'],
               reporte_antes['consistencia_nombres'],reporte_antes['unicidad'],reporte_antes['oportunidad']]
valores_despues=[reporte_despues['completitud'],reporte_despues['exactitud_ruc'],
                 reporte_despues['consistencia_nombres'],reporte_despues['unicidad'],reporte_despues['oportunidad']]

x=range(len(dimensiones))
fig,ax=plt.subplots(figsize=(12,6))
width=0.35
bars1=ax.bar([i-width/2 for i in x],valores_antes,width,label='Antes',color='#C00000',alpha=0.8)
bars2=ax.bar([i+width/2 for i in x],valores_despues,width,label='Despues',color='#1A7F37',alpha=0.8)
ax.set_xlabel('Dimension de Calidad'); ax.set_ylabel('Score (%)')
ax.set_title('BancoSur — Calidad de Datos: Antes vs Despues del Pipeline')
ax.set_xticks(x); ax.set_xticklabels(dimensiones,ha='center')
ax.axhline(90,color='orange',linestyle='--',label='Meta SBS: 90%')
ax.legend(); ax.set_ylim(0,105)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,f'{bar.get_height():.1f}%',ha='center',fontsize=8,fontweight='bold')
plt.tight_layout(); plt.savefig('bancosur_calidad.png',dpi=150,bbox_inches='tight'); plt.show()

## PASO 5: Marco de Gobierno de Datos — DAMA-DMBOK

In [ ]:
marco_gobierno={
    'organizacion':'BancoSur',
    'framework':'DAMA-DMBOK v2',
    'roles':[
        {'rol':'Data Owner','responsable':'Gerente de cada area de negocio',
         'responsabilidades':['Aprobar politicas de calidad de su dominio','Definir reglas de negocio para los datos','Autorizar accesos y permisos']},
        {'rol':'Chief Data Officer (CDO)','responsable':'Marcos Diaz',
         'responsabilidades':['Liderar la estrategia de gobierno de datos','Reportar a la SBS','Coordinar entre areas']},
        {'rol':'Data Steward','responsable':'Analistas designados por area',
         'responsabilidades':['Mantener diccionario de datos actualizado','Resolver problemas de calidad operativos','Documentar linaje de datos']},
        {'rol':'Data Engineer','responsable':'Equipo de TI',
         'responsabilidades':['Construir pipelines ETL','Mantener infraestructura de datos','Implementar controles tecnicos de calidad']},
    ],
    'politicas':[
        {'id':'P-001','nombre':'Politica de Calidad de Datos','descripcion':'Todo dato critico debe tener completitud > 95% y exactitud > 98% antes de ser procesado para reportes SBS.'},
        {'id':'P-002','nombre':'Politica de Gestion de Duplicados','descripcion':'Ningun cliente puede existir con mas de 1 registro activo. La deteccion de duplicados debe ejecutarse semanalmente.'},
        {'id':'P-003','nombre':'Politica de Actualizacion de Datos','descripcion':'Los datos de contacto de clientes deben verificarse anualmente. Registros sin verificacion en >2 anos son marcados como pendientes.'},
    ]
}
import json
print('MARCO DE GOBIERNO DE DATOS BANCOSUR:')
print(json.dumps(marco_gobierno,ensure_ascii=False,indent=2))

## PASO 6: Reporte Final para la SBS

In [ ]:
conn=sqlite3.connect(':memory:')
df_corr.to_sql('clientes_bancosur',conn,if_exists='replace',index=False)

reporte_sbs=pd.DataFrame({
    'dimension':['Completitud','Exactitud RUC','Consistencia Nombres','Unicidad','Oportunidad'],
    'score_inicial':valores_antes,'score_final':valores_despues,
    'mejora_pp':[round(d-a,2) for a,d in zip(valores_antes,valores_despues)],
    'cumple_meta_90':['SI' if d>=90 else 'NO' for d in valores_despues],
})
reporte_sbs.to_sql('reporte_calidad_sbs',conn,if_exists='replace',index=False)
print('REPORTE EJECUTIVO PARA LA SBS:')
display(reporte_sbs)

score_global=round(sum(valores_despues)/len(valores_despues),2)
print(f'\nSCORE GLOBAL DE CALIDAD: {score_global}%')
print(f'ESTADO: {"APROBADO" if score_global>=90 else "REQUIERE ACCION"}')
print(f'Registros corregidos: {len(df_corr)}')
print('Datos y reporte guardados en SQLite.')

## ACTIVIDAD 3: Análisis Reflexivo

**A.** ¿Cuál era la dimensión más crítica para BancoSur? ¿Por qué?

*(Su respuesta aqui)*

---

**B.** ¿Qué porcentaje de mejora logró el pipeline en cada dimensión? ¿Fue suficiente para la SBS?

*(Su respuesta aqui)*

---

**C.** ¿Diferencia entre Data Owner y Data Steward en BancoSur? ¿Quién asumiría cada rol?

*(Su respuesta aqui)*

---

**D.** Si BancoSur adquiere otro banco, ¿qué política de gobierno se activaría de inmediato?

*(Su respuesta aqui)*

## CONCLUSIONES

1. *(Conclusión 1)*

2. *(Conclusión 2)*

3. *(Conclusión 3)*

---
**Docente:** Pilar Rocío Sayán Mejía | **FGD 2026-I** | **LAB-C7**